# Product Category Translation - Bronze Ingestion

## Imports


In [0]:
import uuid
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import StringType, StructField, StructType

## Configuration

In [0]:
environment = "dev"

catalog = f"ecommerce_{environment}"
schema = "bronze"
table_name = "olist_product_category_translation"
target_table = f"{catalog}.{schema}.{table_name}"

source_system = "olist"
source_dataset = "product_category_translation"

source_path = (
    f"/Volumes/{catalog}/landing/raw_files/"
    f"{source_system}/{source_dataset}/"
)

checkpoint_path = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/checkpoint/"
)

schema_location = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/schema/"
)

run_id = str(uuid.uuid4())

## Schema Definition

In [0]:
source_schema = StructType([
    StructField("product_category_name", StringType(), True),
    StructField("product_category_name_english", StringType(), True),
])

## Read with Auto Loader

In [0]:
source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .schema(source_schema)
    .load(source_path)
)

## Add Bronze metadata

In [0]:
bronze_df = (
    source_df
    .withColumn("source_file_path", col("_metadata.file_path"))
    .withColumn("source_file_modification_time", col("_metadata.file_modification_time"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_run_id", lit(run_id))
    .withColumn("source_system", lit(source_system))
    .withColumn("source_dataset", lit(source_dataset))
)

## Write to the Bronze table

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
spark.table(target_table).printSchema()

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(spark.table(target_table).limit(5))

product_category_name,product_category_name_english,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
beleza_saude,health_beauty,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
informatica_acessorios,computers_accessories,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
automotivo,auto,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
cama_mesa_banho,bed_bath_table,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
moveis_decoracao,furniture_decor,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation


In [0]:
spark.table(target_table).count()

71